In [ ]:
import pandas as pd

# PASUL 0 — Citirea datelor din fișierul local
cadru_date_firme = pd.read_csv("lab5.csv", dtype={'caem': str, 'nace_ue': str})

cadru_date_firme['caem'] = cadru_date_firme['caem'].str.strip()
if 'nace_ue' in cadru_date_firme.columns:
    cadru_date_firme['nace_ue'] = cadru_date_firme['nace_ue'].str.strip()

cadru_date_firme['diviziune'] = cadru_date_firme['caem'].str[:2].astype(int)

# PASUL 1 — Secțiune din diviziune, coduri invalide, structură
intervale_sectiuni = [('A',1,3),('B',5,9),('C',10,33),('D',35,35),('E',36,39),
                      ('F',41,43),('G',45,47),('H',49,53),('I',55,56),('J',58,63),
                      ('K',64,66),('L',68,68),('M',69,75),('N',77,82),('O',84,84),
                      ('P',85,85),('Q',86,88),('R',90,93),('S',94,96),('T',97,98),
                      ('U',99,99)]

def determina_sectiunea(diviziune: int):
    for litera_sectiune, limita_inferioara, limita_superioara in intervale_sectiuni:
        if limita_inferioara <= diviziune <= limita_superioara:
            return litera_sectiune
    return None

cadru_date_firme['sectiune'] = cadru_date_firme['diviziune'].map(determina_sectiunea)
coduri_invalide = cadru_date_firme[cadru_date_firme['sectiune'].isna()]
coduri_valide   = cadru_date_firme.dropna(subset=['sectiune'])

sinteza_pe_sectiuni = coduri_valide.groupby('sectiune').agg(
    firme=('id', 'count'),
    angajati=('angajati', 'sum'),
)
sinteza_pe_sectiuni['pondere_%'] = (
    100 * sinteza_pe_sectiuni['angajati'] / sinteza_pe_sectiuni['angajati'].sum()
).round(1)

print("PAS 1 — Structura pe secțiuni")
print(sinteza_pe_sectiuni)
print("\nCoduri invalide:")
print(coduri_invalide[['id', 'denumire', 'caem']].to_string(index=False))

# PASUL 2 — Crosswalk pe baza firmelor din CSV
print("\n")
print("PAS 2 — Crosswalk cu alocare (date reale)")

firme_crosswalk = cadru_date_firme[cadru_date_firme['denumire'].isin([
    'Brutaria Nord', 'Pizzeria Roma', 'Magazin Univers', 'SoftLab'
])][['denumire', 'caem', 'angajati']].copy()

firme_crosswalk = firme_crosswalk.rename(columns={
    'denumire': 'firma',
    'caem': 'cod_intern'
})

tabel_corespondenta = pd.DataFrame({
    'cod_intern': ['10.71', '56.10', '47.11', '62.01', '62.01'],
    'nace':       ['10.71', '56.10', '47.11', '62.01', '62.02'],
    'pondere':    [1, 1, 1, 0.6, 0.4],
})

alocare_combinata = firme_crosswalk.merge(tabel_corespondenta, on='cod_intern')
alocare_combinata['angajati_alocati'] = (
    alocare_combinata['angajati'] * alocare_combinata['pondere']
)

print(alocare_combinata.groupby('nace')['angajati_alocati'].sum())

tip_asociere = tabel_corespondenta.groupby('cod_intern')['nace'].nunique()
print(f"\nTipuri: 1:1 = {(tip_asociere == 1).sum()}, "
      f"1:n = {(tip_asociere > 1).sum()} (cod 62.01)")

total_firme = firme_crosswalk['angajati'].sum()
total_alocat = alocare_combinata['angajati_alocati'].sum()
print(f"Control: {total_firme} = {total_alocat:.0f}")

# PASUL 3 — Indicatori de calitate
publicatii = pd.DataFrame({
    'lucrare':         [f'L{i}' for i in range(1, 13)],
    'intarziere_zile': [0,0,0,2,0,0,5,0,0,0,1,0],
})

procent_punctualitate = 100 * (publicatii['intarziere_zile'] == 0).mean()
rata_raspuns          = 100 * 900 / 2000
completitudine        = 100 * coduri_valide.shape[0] / len(cadru_date_firme)

print("\n")
print("PAS 3 — Indicatori de calitate")
print(f"Punctualitate:   {procent_punctualitate:.1f}%")
print(f"Rata de răspuns: {rata_raspuns:.1f}%")
print(f"Completitudine:  {completitudine:.1f}%")

# EXERCIȚIUL 1 — Firme cu cod 0300 și 9900
print("\n")
print("EXERCIȚIUL 1 — Firme cu cod 0300 și 9900")

cadru_date_firme_cu_coduri_test = pd.concat([
    cadru_date_firme,
    pd.DataFrame({
        'id':         [17, 18],
        'denumire':   ['Firma 0300', 'Firma 9900'],
        'activitate': ['test', 'test'],
        'caem':       ['03.00', '99.00'],
        'angajati':   [10, 20],
    }),
], ignore_index=True)

cadru_date_firme_cu_coduri_test['caem'] = (
    cadru_date_firme_cu_coduri_test['caem'].str.zfill(5)
)
cadru_date_firme_cu_coduri_test['diviziune'] = (
    cadru_date_firme_cu_coduri_test['caem'].str[:2].astype(int)
)
cadru_date_firme_cu_coduri_test['sectiune'] = (
    cadru_date_firme_cu_coduri_test['diviziune'].map(determina_sectiunea)
)

print(cadru_date_firme_cu_coduri_test[
    cadru_date_firme_cu_coduri_test['id'].isin([17, 18])
][['id','caem','diviziune','sectiune']].to_string(index=False))

# EXERCIȚIUL 2 — Ponderi R4: 0.7 / 0.3
print("\n")
print("EXERCIȚIUL 2 — Ponderi R4: 0.7 / 0.3")

tabel_corespondenta_alternativ = tabel_corespondenta.copy()
tabel_corespondenta_alternativ.loc[
    (tabel_corespondenta_alternativ['cod_intern'] == 'R4') &
    (tabel_corespondenta_alternativ['nace'] == '62.01'), 'pondere'
] = 0.7
tabel_corespondenta_alternativ.loc[
    (tabel_corespondenta_alternativ['cod_intern'] == 'R4') &
    (tabel_corespondenta_alternativ['nace'] == '62.02'), 'pondere'
] = 0.3

alocare_combinata_alternativa = registru_firme.merge(
    tabel_corespondenta_alternativ, on='cod_intern'
)
alocare_combinata_alternativa['angajati_alocati'] = (
    alocare_combinata_alternativa['angajati']
    * alocare_combinata_alternativa['pondere']
)

print(alocare_combinata_alternativa.groupby('nace')['angajati_alocati'].sum())
print(f"\nControl: {registru_firme['angajati'].sum()} = "
      f"{alocare_combinata_alternativa['angajati_alocati'].sum():.0f}")

# EXERCIȚIUL 3 — Punctualitate cu toleranță ≤ 1 zi
print("\n")
print("EXERCIȚIUL 3 — Punctualitate (≤ 1 zi = la termen)")

procent_punctualitate_cu_toleranta = 100 * (
    publicatii['intarziere_zile'] <= 1
).mean()
print(f"Punctualitate (≤ 1 zi): {procent_punctualitate_cu_toleranta:.1f}%")
print(f"(față de {procent_punctualitate:.1f}% fără toleranță)")

# EXERCIȚIUL 4 — Principii CoP vizate de fiecare indicator
print("\n")
print("EXERCIȚIUL 4 — Principii CoP și măsuri propuse")
print("""
1. Punctualitate (75.0%) — principiul 13 CoP (actualitate/timeliness)
   Măsură: notificări automate cu 3 zile înainte de termen.

2. Rata de răspuns (45.0%) — principiile 12 (acuratețe) și 9 (relevanță)
   Măsură: simplificarea chestionarului + canal online de colectare.

3. Completitudine cod CAEM (93.8%) — principiile 12 (acuratețe)
   și 14 (completitudine)
   Măsură: validare automată a codului la introducere + listă de sugestii.
""")


PAS 1 — Structura pe secțiuni
          firme  angajati  pondere_%
sectiune                            
A             2        30        2.4
C            12       369       29.1
F             1       150       11.8
G             1       120        9.5
H             1        35        2.8
I             2        82        6.5
J             2       107        8.5
M             1         6        0.5
P             1        48        3.8
Q             2       319       25.2

Coduri invalide:
 id denumire  caem
 16  Firma X 44.00


PAS 2 — Crosswalk cu alocare (date reale)
nace
10.71     42.0
47.11    120.0
56.10     28.0
62.01     51.0
62.02     34.0
Name: angajati_alocati, dtype: float64

Tipuri: 1:1 = 3, 1:n = 1 (cod 62.01)
Control: 275 = 275


PAS 3 — Indicatori de calitate
Punctualitate:   75.0%
Rata de răspuns: 45.0%
Completitudine:  96.2%


EXERCIȚIUL 1 — Firme cu cod 0300 și 9900
 id  caem  diviziune sectiune
 17 10.32         10        C
 18 10.39         10        C
 17 03.00      